In [0]:
from pyspark.sql.functions import *

In [0]:
events_df = spark.read.table("e_commerce_project.gold.events")
order_items_df = spark.read.table("e_commerce_project.gold.order_items")
orders_df = spark.read.table("e_commerce_project.gold.orders")
products_df = spark.read.table("e_commerce_project.gold.products")
reviews_df = spark.read.table("e_commerce_project.gold.reviews")
users_df = spark.read.table("e_commerce_project.gold.users")

In [0]:
events_df.limit(5).display()

event_id,user_id,product_id,event_type,event_timestamp
E00000001,U009798,P001393,cart,2025-07-08T14:28:55.893Z
E00000002,U005881,P000669,view,2025-10-19T23:00:44.067Z
E00000003,U006348,P001404,view,2025-05-09T07:02:42.256Z
E00000004,U002664,P000400,cart,2025-07-19T22:47:07.019Z
E00000005,U005776,P000392,view,2024-10-24T10:20:33.602Z


**Checking for Distinct Values in Event Type Column**

In [0]:
events_df.select("event_type").distinct().display()

event_type
cart
view
wishlist
purchase


**Seperated the data based on user id, whether the user viewed, purchased, wishlisted or added to cart**


In [0]:
user_events = events_df.groupby("user_id").pivot("event_type").count().fillna(0)

**If the user purchased it will give the count as 1, otherwise 0**

In [0]:
ml_data = user_events.withColumn("Target",when(col("purchase")>0,1).otherwise(0))

**Converting the dataframe into Pandas**

In [0]:
df = ml_data.toPandas()

**Train/Test Split**

In [0]:
from sklearn.model_selection import train_test_split

X = df[['view','cart','wishlist']]
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
print(X_train.shape)
print(X_test.shape)

(7996, 3)
(1999, 3)


**Model Training**

In [0]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [0]:
pred = model.predict(X_test)

**Model Evaluation**

In [0]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

In [0]:
print("Model Accuracy:", accuracy)

Model Accuracy: 0.6548274137068534


In [0]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)

In [0]:
print(cm)

[[1295   33]
 [ 657   14]]


In [0]:
import pandas as pd
importance = pd.Series(model.feature_importances_, index=X.columns)


In [0]:
print(importance)

view        0.560450
cart        0.235391
wishlist    0.204158
dtype: float64


**Classification Report**

In [0]:
from sklearn.metrics import classification_report


In [0]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.66      0.98      0.79      1328
           1       0.30      0.02      0.04       671

    accuracy                           0.65      1999
   macro avg       0.48      0.50      0.41      1999
weighted avg       0.54      0.65      0.54      1999

